In [ ]:
import os
from pathlib import Path
from google.colab import drive

# 1. Montar Drive
drive.mount("/content/drive")

# 2. Buscar automáticamente el archivo en MyDrive
archivos = list(Path("/content/drive/MyDrive").rglob("etiquetas_palabras.csv"))

if archivos:
    carpeta_tp = archivos[0].parent
    os.environ["RF_DATA_DIR"] = str(carpeta_tp)
    print("Archivo encontrado en:", carpeta_tp)
else:
    print("No se encontró 'etiquetas_palabras.csv' en MyDrive.")
    print("Si la carpeta es compartida, agregale un acceso directo a 'Mi unidad' en Google Drive.")

In [ ]:
import argparse
import json
import os
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
SEED_TEST = 42
SEED_VALIDACION = 2026
SEED_MODELO = 42
TAREAS = ["punt_inicial", "punt_final", "capitalizacion"]  # nombres internos
COL_CAP_SALIDA = "capitalización"  # nombre de la columna en el CSV de entrega (ver PDF)
VENTANA = 2
TOP_K_PALABRAS = 150

INTERROGATIVOS = {
    "que", "qué", "quien", "quién", "quienes", "quiénes", "cual", "cuál",
    "cuales", "cuáles", "cuando", "cuándo", "donde", "dónde", "como", "cómo",
    "cuanto", "cuánto", "cuanta", "cuánta", "cuantos", "cuántos", "cuantas", "cuántas",
}
CONECTORES_COMA = {
    "pero", "aunque", "sino", "ademas", "además", "entonces", "embargo",
    "osea", "decir", "ejemplo", "porque", "mientras", "pues",
}
ARTICULOS_PREP = {"el", "la", "los", "las", "un", "una", "de", "del", "en", "a", "y", "o"}
VOCALES = set("aeiouáéíóúü")

# Tokens BERT
ESPECIALES = {"[CLS]", "[SEP]", "[PAD]", "[UNK]", "[MASK]"}
APOSTROFES = {"'", "’"}
SUFIJOS_APOSTROFE = {"s", "t", "re", "ve", "ll", "d", "m"}  # McDonald ' s, don ' t, ...

In [ ]:
def encontrar_data_dir(arg_datos=None) -> Path:
    ubicaciones = []
    if arg_datos:
        ubicaciones.append(Path(arg_datos))
    if os.environ.get("RF_DATA_DIR"):
        ubicaciones.append(Path(os.environ["RF_DATA_DIR"]))
    ubicaciones += [Path.cwd(), Path.cwd() / "salidas", Path.cwd() / "datos_modelo",
                    Path("/content/drive/MyDrive/TP_AA")]
    d = next((p for p in ubicaciones if (p / "etiquetas_palabras.csv").is_file()), None)
    if d is None:
        raise FileNotFoundError("No se encontró etiquetas_palabras.csv. Usá --datos o RF_DATA_DIR.")
    return d


def cargar_etiquetas(data_dir: Path) -> pd.DataFrame:
    y = pd.read_csv(data_dir / "etiquetas_palabras.csv", keep_default_na=False)
    y = y.rename(columns={"capitalización": "capitalizacion"})  # unifica tilde/sin tilde

    col_texto = next((c for c in ["palabra", "token", "texto", "word"] if c in y.columns), None)
    if col_texto is not None:
        palabras = y[col_texto].astype(str).tolist()
    else:
        print("Reconstruyendo palabras desde embeddings_palabras.py...")
        sys.path.insert(0, str(data_dir))
        from embeddings_palabras import cargar_fragmentos_filtrados, palabras_y_etiquetas
        df_frag = cargar_fragmentos_filtrados()
        mapa_frag = dict(zip(df_frag["Indice"].astype(int), df_frag["Fragmento"]))
        palabras = []
        for inst_id in y["instancia_id"].unique():
            items = palabras_y_etiquetas(mapa_frag[int(inst_id)])
            palabras.extend(str(it[0]) for it in items)
    assert len(palabras) == len(y), f"Desajuste: {len(palabras)} palabras vs {len(y)} filas"

    y = y.reset_index(drop=True)
    y["_orig"] = palabras                              # con su capitalización original
    y["_norm"] = y["_orig"].str.lower().str.strip()    # lo que ven los atributos
    y["punt_inicial"] = y["punt_inicial"].astype(str)
    y["punt_final"] = y["punt_final"].astype(str)
    y["capitalizacion"] = y["capitalizacion"].astype(int)

    inst = y["instancia_id"].to_numpy()
    if (inst[1:] != inst[:-1]).sum() + 1 != len(np.unique(inst)):
        raise ValueError("Las filas de una misma instancia_id deben ser contiguas.")
    return y


def split_del_grupo(y: pd.DataFrame):
    """Mismo split que el notebook: test 15% (SEED_TEST), val 15% del total (SEED_VALIDACION)."""
    grupos = y["instancia_id"].to_numpy()
    X_dummy = np.zeros((len(y), 1))
    gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED_TEST)
    train_val_idx, test_idx = next(gss_test.split(X_dummy, groups=grupos))
    gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15 / 0.85, random_state=SEED_VALIDACION)
    tr_rel, val_rel = next(gss_val.split(X_dummy[train_val_idx], groups=grupos[train_val_idx]))
    return train_val_idx[tr_rel], train_val_idx[val_rel], test_idx

In [ ]:
def atributos_palabra(p: str, pos: int, total: int) -> list:
    largo = len(p)
    vocales = sum(c in VOCALES for c in p)
    return [
        largo, vocales,
        sum(c.isalpha() and c not in VOCALES for c in p),
        sum(c in "áéíóú" for c in p),
        sum(c.isdigit() for c in p),
        int("'" in p or "’" in p),
        int("-" in p),
        int(p.isalpha()),
        vocales / max(largo, 1),
        int(p in INTERROGATIVOS),
        int(p in CONECTORES_COMA),
        int(p in ARTICULOS_PREP),
        int(pos == 0),
        int(pos == total - 1),
        pos / max(total - 1, 1),
    ]


def desplazar(M: np.ndarray, off: int, inst: np.ndarray, fill):
    """M[i+off] si pertenece a la misma instancia que i; si no, `fill`."""
    n = len(M)
    idx = np.arange(n) + off
    ok = (idx >= 0) & (idx < n)
    idx = np.clip(idx, 0, n - 1)
    ok &= inst[idx] == inst
    out = M[idx].copy()
    out[~ok] = fill
    return out


def construir_atributos(df: pd.DataFrame, vocab: list) -> sparse.csr_matrix:
    """df: columnas `instancia_id` (contiguas) y `_norm`. Devuelve matriz esparsa."""
    inst = df["instancia_id"].to_numpy()
    palabras = df["_norm"].tolist()
    n = len(df)
    pos = df.groupby("instancia_id", sort=False).cumcount().to_numpy()
    total = df.groupby("instancia_id", sort=False)["_norm"].transform("size").to_numpy()
    base = np.array([atributos_palabra(p, int(i), int(t))
                     for p, i, t in zip(palabras, pos, total)], dtype=np.float32)

    idx_vocab = {w: i for i, w in enumerate(vocab)}
    cod = np.array([idx_vocab.get(p, -1) for p in palabras], dtype=np.int64)
    K = len(vocab)

    densos, lexicos = [], []
    for off in range(-VENTANA, VENTANA + 1):
        densos.append(desplazar(base, off, inst, -1.0))
        sh = desplazar(cod, off, inst, -1)
        filas = np.where(sh >= 0)[0]
        lexicos.append(sparse.csr_matrix(
            (np.ones(len(filas), dtype=np.float32), (filas, sh[filas])), shape=(n, K)))
    X_dense = sparse.csr_matrix(np.hstack(densos))
    return sparse.hstack([X_dense] + lexicos, format="csr", dtype=np.float32)


In [ ]:
def armar_dict_mixtas(df_train: pd.DataFrame) -> dict:
    """{'mcdonald's': "McDonald's", 'iphone': 'iPhone'}; solo con palabras del TRAIN."""
    cont = defaultdict(Counter)
    m = df_train[df_train["capitalizacion"] == 2]
    for orig, norm in zip(m["_orig"], m["_norm"]):
        cont[norm][orig.strip()] += 1
    d = {k: v.most_common(1)[0][0] for k, v in cont.items()}
    if d and not any(any(c.isupper() for c in w[1:]) for w in d.values()):
        print("ADVERTENCIA: las palabras de capitalización 2 no tienen mayúsculas internas; "
              "parece que la columna de palabras perdió la capitalización original.")
    return d


def aplicar_cap(w: str, cap, dict_mixtas: dict) -> str:
    cap = int(cap)
    if cap == 3:
        return w.upper()
    if cap == 2:
        return dict_mixtas.get(w) or (w[:1].upper() + w[1:])  # si no está, mejor estimación
    if cap == 1:
        return w[:1].upper() + w[1:]
    return w


In [ ]:
def entrenar(args):
    data_dir = encontrar_data_dir(args.datos)
    salida = Path(args.salida)
    salida.mkdir(parents=True, exist_ok=True)

    y = cargar_etiquetas(data_dir)
    train_idx, val_idx, test_idx = split_del_grupo(y)
    print(f"Datos: {data_dir} | Split -> Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

    vocab = [w for w, _ in Counter(y.iloc[train_idx]["_norm"]).most_common(TOP_K_PALABRAS)]
    dict_mixtas = armar_dict_mixtas(y.iloc[train_idx])
    print(f"Vocabulario one-hot: {len(vocab)} palabras | dict_mixtas: {len(dict_mixtas)} entradas")

    t0 = time.time()
    X = construir_atributos(y, vocab)
    print(f"Matriz de atributos: {X.shape} ({time.time() - t0:.1f}s)")
    sparse.save_npz(salida / "X_atributos.npz", X)

    modelos, metricas = {}, {}
    for tarea in TAREAS:
        print(f"\n{'=' * 50}\nRandom Forest (atributos) -> {tarea}\n{'=' * 50}")
        t1 = time.time()
        clf = RandomForestClassifier(
            n_estimators=150, max_depth=25, min_samples_leaf=2,
            class_weight="balanced", n_jobs=-1, random_state=SEED_MODELO)
        clf.fit(X[train_idx], y.iloc[train_idx][tarea].to_numpy())
        modelos[tarea] = clf

        real = y.iloc[val_idx][tarea].to_numpy()
        pred = clf.predict(X[val_idx])
        f1 = f1_score(real, pred, average="macro")
        metricas[tarea] = {"f1_macro_val": float(f1)}
        print(f"Tiempo: {time.time() - t1:.1f}s | F1-macro (val): {f1:.4f}")
        print(classification_report(real, pred, digits=4, zero_division=0))

        if args.test:
            real_t = y.iloc[test_idx][tarea].to_numpy()
            f1_t = f1_score(real_t, clf.predict(X[test_idx]), average="macro")
            metricas[tarea]["f1_macro_test"] = float(f1_t)
            print(f"F1-macro (TEST): {f1_t:.4f}")

    joblib.dump({"modelos": modelos, "vocab": vocab, "dict_mixtas": dict_mixtas},
                salida / "modelo_rf.joblib", compress=3)
    (salida / "metricas_rf_atributos.json").write_text(json.dumps(metricas, indent=2))
    print("\nGuardado en:", salida.resolve())


In [ ]:
def cargar_modelo(ruta) -> dict:
    return joblib.load(ruta)


def predecir_palabras(palabras: list, pack: dict) -> dict:
    """palabras: lista de palabras (una instancia). Devuelve dict tarea -> array de etiquetas."""
    df = pd.DataFrame({"instancia_id": 0, "_norm": [p.lower().strip() for p in palabras]})
    X = construir_atributos(df, pack["vocab"])
    return {t: pack["modelos"][t].predict(X) for t in TAREAS}


def agrupar_tokens(tokens: list, unir_apostrofe: bool = True) -> list:
    """Agrupa tokens BERT en palabras. Devuelve lista de listas de índices de tokens.
    - '##xxx' se une a la palabra anterior.
    - Con unir_apostrofe: "mcdonald ' s" (-> m ##c ##dona ##ld ' s) se une como una palabra."""
    grupos = []
    for i, t in enumerate(tokens):
        if t in ESPECIALES:
            continue
        une = False
        if grupos:
            ult = tokens[grupos[-1][-1]]
            sig = tokens[i + 1] if i + 1 < len(tokens) else None
            une = t.startswith("##")
            if unir_apostrofe and not une:
                if t in APOSTROFES and sig in SUFIJOS_APOSTROFE:
                    une = True
                elif ult in APOSTROFES and t in SUFIJOS_APOSTROFE and len(grupos[-1]) > 1:
                    une = True
        if une:
            grupos[-1].append(i)
        else:
            grupos.append([i])
    return grupos


def predecir_tokens(tokens: list, pack: dict, unir_apostrofe: bool = True) -> pd.DataFrame:
    """tokens: tokens BERT de UNA instancia. Devuelve un DataFrame con una fila por token
    y columnas punt_inicial, punt_final, capitalización (formato de la cátedra).
      - punt_inicial   -> solo en el PRIMER sub-token de la palabra
      - punt_final     -> solo en el ÚLTIMO sub-token de la palabra
      - capitalización -> repetida en TODOS los sub-tokens de la palabra"""
    n = len(tokens)
    ini, fin, cap = [""] * n, [""] * n, [0] * n
    grupos = agrupar_tokens(tokens, unir_apostrofe)
    if grupos:
        palabras = ["".join(tokens[i][2:] if tokens[i].startswith("##") else tokens[i] for i in g)
                    for g in grupos]
        pred = predecir_palabras(palabras, pack)
        for k, g in enumerate(grupos):
            ini[g[0]] = pred["punt_inicial"][k]
            fin[g[-1]] = pred["punt_final"][k]
            for i in g:
                cap[i] = int(pred["capitalizacion"][k])
    return pd.DataFrame({"token": tokens, "punt_inicial": ini, "punt_final": fin,
                         COL_CAP_SALIDA: cap})


def restaurar_texto(texto: str, pack: dict) -> str:
    """Demo: texto plano sin puntuación -> texto restaurado."""
    palabras = [p.lower() for p in texto.split()]
    if not palabras:
        return ""
    pred = predecir_palabras(palabras, pack)
    out, mayus = [], True
    for i, w in enumerate(palabras):
        cap = int(pred["capitalizacion"][i])
        if mayus and cap == 0:
            cap = 1
        out.append(pred["punt_inicial"][i] + aplicar_cap(w, cap, pack["dict_mixtas"]) + pred["punt_final"][i])
        mayus = pred["punt_final"][i] in (".", "?")
    return " ".join(out)

In [ ]:
if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--datos", help="Carpeta con etiquetas_palabras.csv")
    ap.add_argument("--salida", default="resultados_rf_atributos")
    ap.add_argument("--test", action="store_true", help="Evaluar también en test (usalo al final)")
    ap.add_argument("--restaurar", help="Texto plano sin puntuación")
    ap.add_argument("--tokens", help="Tokens BERT separados por espacio (una instancia)")

    # Usamos parse_known_args en lugar de parse_args para ignorar los argumentos internos del kernel de Colab
    a, _ = ap.parse_known_args()

    ruta_modelo = Path(a.salida) / "modelo_rf.joblib"
    if a.restaurar is not None:
        print(restaurar_texto(a.restaurar, cargar_modelo(ruta_modelo)))
    elif a.tokens is not None:
        print(predecir_tokens(a.tokens.split(), cargar_modelo(ruta_modelo)).to_string())
    else:
        entrenar(a)